# 01 · Datos crudos y transformación (Bloque 1)

Este notebook trabaja con **precios** y tiene partes con reglas distintas:

| Parte | Datos que usa | Para qué | ¿Decide el diseño? |
|---|---|---|---|
| **A. Auditoría de calidad** | Todo el rango 2010-2026 | nulos, huecos, valores ≤ 0, calendario, saltos | No: solo detecta errores de datos |
| **B. Partición** | — | verificar los cortes train / val / test definidos en `src/config.py` | — |
| **C. Diagnóstico de transformaciones** | **Solo train** | elegir log-retorno o diferencia por serie | **Sí** |
| **D. Retornos y guardado** | Toda la serie (transformación punto a punto, sin parámetros) | genera `data/processed/` | — |

Los notebooks **02** y **03** solo leen `data/processed/` y nunca abren el test.

In [1]:
import sys
from pathlib import Path

# La raíz del proyecto es la carpeta que contiene `src/` (funciona se ejecute desde la raíz o desde notebooks/)
RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

In [2]:
RAIZ

WindowsPath('c:/Users/ASUS/Desktop/Maestria IA USFQ/TESIS/DESARROLLO TESIS')

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import importlib

from src import config as cfg

from src import load_clean as lc
from src.preprocessing import construir_retornos, preparar_y_guardar, cargar
from src import eda_utils as eda

In [4]:
importlib.reload(cfg)
importlib.reload(lc)

<module 'src.load_clean' from 'c:\\Users\\ASUS\\Desktop\\Maestria IA USFQ\\TESIS\\DESARROLLO TESIS\\src\\load_clean.py'>

## 1. Revisiòn de la calidad de la data

In [5]:
base = lc.cargar_crudo()
print(base.shape, base.index.min().date(), "->", base.index.max().date())
base.head()

(4232, 7) 2010-01-04 -> 2026-10-20


,WTI,VIX,DXY,GOLD,TNX,SP500,NG
Date,,,,,,,
2010-01-04,81.510002,20.040001,77.529999,1117.699951,3.841,1132.989990,5.884
2010-01-05,81.769997,19.350000,77.620003,1118.099976,3.755,1136.520020,5.637
2010-01-06,83.180000,19.160000,77.489998,1135.900024,3.808,1137.140015,6.009
2010-01-07,82.660004,19.059999,77.910004,1133.099976,3.822,1141.689941,5.806
2010-01-08,82.750000,18.129999,77.470001,1138.199951,3.808,1144.979980,5.749


Se extraen  los precios al cierre de las variables consideradas dentro del proyecto, con 4232 observaciones desde el 04 de Enero del 2010 al 20 de Octubre de 2026.

Las variables a considerar son las siguientes:



| Variable | Unidades | Descripción |
|---|---|---|
| `Date` | Fecha (AAAA-MM-DD) | Fecha de observación de las variables financieras. Permite ordenar cronológicamente las series, sincronizar los datos y estudiar la evolución de los mercados financieros a lo largo del tiempo. |
| `WTI` | USD por barril (USD/bbl) | Precio internacional del petróleo West Texas Intermediate, utilizado como referencia del mercado petrolero estadounidense.  Un incremento del precio puede favorecer los ingresos petroleros, la balanza comercial y la disponibilidad de recursos fiscales, mientras que una caída puede reducirlos y aumentar las necesidades de financiamiento público. |
| `VIX` | Puntos de índice (10 a 40 puntos) | Índice de volatilidad implícita del S&P 500, que representa las expectativas del mercado sobre la volatilidad del mercado bursátil estadounidense durante los próximos 30 días, expresadas en términos anualizados. El S&P 500 agrupa aproximadamente 500 grandes empresas cotizadas en Estados Unidos. Un aumento del VIX suele reflejar mayor incertidumbre y aversión al riesgo entre los inversionistas, mientras que una disminución suele asociarse con mayor tranquilidad en los mercados. |
| `DXY` | Puntos de índice (70 a 120 puntos) | Índice del dólar estadounidense frente a una cesta de seis monedas principales. Su evolución es especialmente relevante para Ecuador debido a que el dólar es su moneda oficial. Una apreciación del dólar puede encarecer relativamente los productos ecuatorianos frente a los de países cuyas monedas se deprecian, afectando potencialmente la competitividad de las exportaciones no petroleras.  |
| `GOLD` | USD por onza troy (USD/oz t) | Precio internacional del oro expresado en dólares estadounidenses. El oro suele utilizarse como activo de inversión y reserva de valor, y puede beneficiarse de episodios de incertidumbre económica, tensiones geopolíticas o cambios en las tasas de interés reales. |
| `TNX` | Porcentaje (%) | Rendimiento del bono del Tesoro estadounidense a 10 años, indicador de las tasas de interés de largo plazo y de las condiciones financieras internacionales. Para Ecuador, al ser una economía dolarizada, es relevante porque las tasas estadounidenses influyen en el costo de referencia del financiamiento en dólares. Un incremento del TNX puede encarecer el financiamiento internacional para el sector público y privado ecuatoriano, especialmente si coincide con un aumento del riesgo país. |






In [6]:
# Nulos por columna y filas afectadas
display(lc.resumen_nulos(base))
base[base.isna().any(axis=1)]

,cantidad,porcentaje
TNX,9,0.21
GOLD,7,0.17
WTI,7,0.17
SP500,7,0.17
DXY,6,0.14
NG,6,0.14
VIX,5,0.12


,WTI,VIX,DXY,GOLD,TNX,SP500,NG
Date,,,,,,,
2010-10-11,82.209999,18.959999,77.440002,1353.300049,NaN,1165.319946,3.601
2016-01-18,NaN,NaN,99.120003,NaN,NaN,NaN,NaN
2016-10-10,NaN,13.380000,NaN,NaN,1.723,2163.659912,NaN
2016-11-11,NaN,14.170000,NaN,NaN,NaN,2164.449951,NaN
2018-01-29,65.559998,13.840000,89.309998,NaN,2.699,2853.530029,3.631
2023-11-23,NaN,NaN,NaN,1993.300049,NaN,NaN,2.934
2025-01-09,73.919998,NaN,109.180000,2690.800049,NaN,NaN,3.701
2025-05-26,NaN,NaN,98.930000,NaN,NaN,NaN,NaN
2025-07-04,66.500000,NaN,NaN,3346.500000,NaN,NaN,3.387
